# RMFS Baseline Pipeline (No Re-Clustering)

Single continuous simulation run with initial k-means clustering only. Use this as a control to compare against the re-clustering pipeline.

## Configuration

In [ ]:
TOTAL_HOURS = 0.25   # Total simulation duration in hours
K = 5                # Number of k-means clusters

# Item cluster order frequency configuration (must sum to 1.0)
ITEMS_ORDERS_CLASS_CONFIG = {
    4: 0.45,  # 45%
    0: 0.25,  # 25%
    2: 0.20,  # 20%
    1: 0.10,  # 10%
    3: 0.05,  #  5%
}

## Setup

In [2]:
import os, sys, io, contextlib, time
import pandas as pd
import numpy as np
from math import ceil, sqrt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from tqdm.notebook import tqdm

# Resolve project paths
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
PIPELINE_DIR = os.path.join(PROJECT_ROOT, "pipeline")
NETLOGO_DIR = os.path.join(PROJECT_ROOT, "netlogo")

sys.path.insert(0, PIPELINE_DIR)
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, NETLOGO_DIR)

# Constants
LEAD_TIME = 1.0
SERVICE_LEVEL_Z = 1.2816
RANDOM_STATE = 42
N_INIT = 20
CLUSTER_FEATURES = ["mean_demand", "cv_demand", "demand_frequency", "avg_affinity", "max_affinity"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Baseline: {TOTAL_HOURS}h, K={K} — single run, no re-clustering")

Project root: /Users/brendantm/Taiwan/TEEP/salsa-rmfs
Baseline: 0.25h, K=5 — single run, no re-clustering


In [3]:
@contextlib.contextmanager
def suppress_stdout():
    old = sys.stdout
    sys.stdout = io.StringIO()
    try:
        yield
    finally:
        sys.stdout = old


def run_phase(target_tick, phase_name):
    """Run tick() loop with a tqdm progress bar."""
    from netlogo import tick as sim_tick

    tick_count = 0
    current_tick = 0.0
    metrics_log = []
    t0 = time.time()

    pbar = tqdm(total=int(target_tick), desc=phase_name, unit="s", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}s [{elapsed}<{remaining}, {postfix}]")

    while current_tick < target_tick:
        with suppress_stdout():
            result = sim_tick()

        if isinstance(result, str):
            print(f"ERROR: {result}")
            break

        current_tick = result[-1]
        tick_count += 1

        metrics_log.append({
            "tick": current_tick,
            "total_energy": result[1],
            "job_queue_len": result[2],
            "stop_and_go": result[3],
            "total_turning": result[4],
            "orders_finished": result[6],
        })

        pbar.update(int(current_tick) - pbar.n)
        if tick_count % 50 == 0:
            pbar.set_postfix(orders=result[6])

    pbar.update(int(target_tick) - pbar.n)
    pbar.set_postfix(orders=metrics_log[-1]["orders_finished"] if metrics_log else 0)
    pbar.close()

    elapsed = time.time() - t0
    metrics_df = pd.DataFrame(metrics_log)
    print(f"  {phase_name} complete: {tick_count} ticks in {elapsed:.1f}s")

    return {"final_tick": current_tick, "tick_count": tick_count, "metrics": metrics_df}


def compute_extra_metrics(orders_finished, generated_order_path, pod_info_path):
    metrics = {}
    orders_generated = 0
    if os.path.exists(generated_order_path):
        gen_df = pd.read_csv(generated_order_path, dtype=str)
        orders_generated = gen_df["order_id"].nunique()
    metrics["orders_generated"] = orders_generated
    metrics["order_throughput"] = orders_finished / orders_generated if orders_generated > 0 else 0.0

    total_picks = total_replenishments = total_units_picked = pod_visits = 0
    if os.path.exists(pod_info_path):
        pod_df = pd.read_csv(pod_info_path)
        if not pod_df.empty and "task_type" in pod_df.columns:
            pick_df = pod_df[pod_df["task_type"] == 1]
            replen_df = pod_df[pod_df["task_type"] == 2]
            total_picks = len(pick_df)
            total_replenishments = len(replen_df)
            total_units_picked = pick_df["qty"].sum() if "qty" in pick_df.columns else 0
            if not pick_df.empty and "pod_id" in pick_df.columns and "processed_time" in pick_df.columns:
                pod_visits = pick_df.groupby(["pod_id", "processed_time"]).ngroups

    metrics["total_picks"] = total_picks
    metrics["total_replenishments"] = total_replenishments
    metrics["replenishment_pick_ratio"] = total_replenishments / total_picks if total_picks > 0 else 0.0
    metrics["total_units_picked"] = total_units_picked
    metrics["pod_visits"] = pod_visits
    metrics["pod_utilization"] = total_units_picked / pod_visits if pod_visits > 0 else 0.0
    return metrics

print("Helper functions defined.")

Helper functions defined.


## Phase 0: Initial K-Means Clustering

In [4]:
sku_sample_path = os.path.join(PROJECT_ROOT, "sku_sample.csv")

sku_df = pd.read_csv(sku_sample_path)
sku_df["item_code"] = sku_df["item_code"].astype(str)
for col in CLUSTER_FEATURES:
    sku_df[col] = pd.to_numeric(sku_df[col], errors="coerce").fillna(0)

X = sku_df[CLUSTER_FEATURES].copy()
X["mean_demand"] = np.log1p(X["mean_demand"])
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
km = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=N_INIT)
sku_df["cluster"] = km.fit_predict(X_scaled)
sku_df.to_csv(sku_sample_path, index=False)

dist = sku_df["cluster"].value_counts().sort_index()
print(f"Clustered {len(sku_df)} SKUs into {K} groups:")
for cluster_id, count in dist.items():
    print(f"  Cluster {cluster_id}: {count} SKUs")

Clustered 1448 SKUs into 5 groups:
  Cluster 0: 414 SKUs
  Cluster 1: 241 SKUs
  Cluster 2: 259 SKUs
  Cluster 3: 59 SKUs
  Cluster 4: 475 SKUs


## Initialize Simulation & Generate Orders

In [ ]:
total_seconds = TOTAL_HOURS * 3600

os.chdir(NETLOGO_DIR)
sku_sample_rel = os.path.relpath(sku_sample_path, NETLOGO_DIR)

# Generate items.csv and pods.csv via thesis-based FFD assignment
from assignment.convert_to_sim import run_full_conversion
run_full_conversion(
    base_dir=PROJECT_ROOT,
    netlogo_dir=NETLOGO_DIR,
    sku_sample_path=sku_sample_path,
)

from netlogo import reload_data_for_phase

total_order_hours = max(1, int(np.ceil(TOTAL_HOURS)))
backlog_order_hours = total_order_hours + 1

print(f"Generating orders for {total_order_hours}h and initializing simulation...")
with suppress_stdout():
    reload_data_for_phase(
        sku_sample_path=sku_sample_rel,
        order_period_hours=total_order_hours,
        backlog_period_hours=backlog_order_hours,
        items_orders_class_configuration=ITEMS_ORDERS_CLASS_CONFIG
    )
print("Simulation initialized.")

## Run Simulation

In [6]:
run_data = run_phase(total_seconds, "Baseline")

Baseline:   0%|          | 0/900s [00:00<?, ]

  Baseline complete: 6001 ticks in 119.1s


## Results

In [7]:
# Save results
results_dir = os.path.join(PROJECT_ROOT, "results_baseline")
os.makedirs(results_dir, exist_ok=True)

order_finished = None
if os.path.exists("order-finished.csv"):
    order_finished = pd.read_csv("order-finished.csv")
    order_finished.to_csv(os.path.join(results_dir, "orders.csv"), index=False)

# Compute extra metrics
m = run_data["metrics"]
finished_count = int(m["orders_finished"].iloc[-1]) if not m.empty else 0
extra = compute_extra_metrics(finished_count, "generated_order.csv", "pod_info.csv")

# Save tick metrics & summary
if not m.empty:
    m["phase"] = "baseline"
    m.to_csv(os.path.join(results_dir, "tick_metrics.csv"), index=False)

summary = {
    "total_hours": TOTAL_HOURS, "k_clusters": K, "pipeline": "baseline",
    "ticks": run_data["tick_count"],
    "orders_finished": m["orders_finished"].iloc[-1] if not m.empty else 0,
    "total_energy": m["total_energy"].iloc[-1] if not m.empty else 0,
    "stop_and_go": m["stop_and_go"].iloc[-1] if not m.empty else 0,
    "total_turning": m["total_turning"].iloc[-1] if not m.empty else 0,
    "peak_job_queue": m["job_queue_len"].max() if not m.empty else 0,
    "avg_job_queue": round(m["job_queue_len"].mean(), 1) if not m.empty else 0,
    "orders_generated": extra["orders_generated"],
    "order_throughput": round(extra["order_throughput"], 4),
    "total_picks": extra["total_picks"],
    "total_replenishments": extra["total_replenishments"],
    "replenishment_pick_ratio": round(extra["replenishment_pick_ratio"], 4),
    "total_units_picked": extra["total_units_picked"],
    "pod_visits": extra["pod_visits"],
    "pod_utilization": round(extra["pod_utilization"], 4),
}
pd.DataFrame([summary]).to_csv(os.path.join(results_dir, "summary.csv"), index=False)

# Display results
print("=" * 60)
print(f"  RESULTS — Baseline {TOTAL_HOURS}h, K={K}")
print("=" * 60)
if not m.empty:
    print(f"    Orders finished:    {m['orders_finished'].iloc[-1]}")
    print(f"    Total energy:       {m['total_energy'].iloc[-1]:.2f}")
    print(f"    Stop & go:          {m['stop_and_go'].iloc[-1]}")
    print(f"    Total turning:      {m['total_turning'].iloc[-1]}")
    print(f"    Peak job queue:     {m['job_queue_len'].max()}")
    print(f"    Avg job queue:      {m['job_queue_len'].mean():.1f}")
if order_finished is not None and not order_finished.empty:
    if "order_complete_time" in order_finished.columns and "process_start_time" in order_finished.columns:
        ct = order_finished["order_complete_time"] - order_finished["process_start_time"]
        print(f"    Avg cycle time:     {ct.mean():.1f}s")
        print(f"    Max cycle time:     {ct.max():.1f}s")
print(f"    Order throughput:   {extra['order_throughput']:.4f} ({extra['orders_generated']} generated)")
print(f"    Replen/pick ratio:  {extra['replenishment_pick_ratio']:.4f} ({extra['total_replenishments']}R / {extra['total_picks']}P)")
print(f"    Pod utilization:    {extra['pod_utilization']:.4f} ({extra['total_units_picked']} units / {extra['pod_visits']} visits)")
print(f"\nResults saved to {results_dir}/")

  RESULTS — Baseline 0.25h, K=5
    Orders finished:    36
    Total energy:       2049251.78
    Stop & go:          2376
    Total turning:      1831
    Peak job queue:     47
    Avg job queue:      29.6
    Order throughput:   0.2130 (169 generated)
    Replen/pick ratio:  0.0244 (3R / 123P)
    Pod utilization:    7.5288 (783 units / 104 visits)

Results saved to /Users/brendantm/Taiwan/TEEP/salsa-rmfs/results_baseline/
